In [ ]:
import ollama
import json
import re

# -----------------------------
# 1. CALL LLM
# -----------------------------
def call_llm(user_input):
    prompt = f"""
You are a system dynamics expert.

Convert the following causal relationship into stock and flow components.

Rules:
1. Identify stocks (something that accumulate over time)
2. 
3. Inflows increase stock
4. Outflows decrease stock
5. Output ONLY valid JSON

Format:
{{
  "stock": "",
  "inflow": [],
  "outflow": [],
  "auxiliary": [],
  "equations": {{}}
}}

Text:
\"\"\"{user_input}\"\"\"
"""

    response = ollama.chat(
        model='qwen3.5',
        messages=[{"role": "user", "content": prompt}]
    )

    return response['message']['content']


# -----------------------------
# 2. PARSE JSON SAFELY
# -----------------------------
def parse_json(response_text):
    try:
        # extract JSON from messy output
        json_match = re.search(r'\{.*\}', response_text, re.DOTALL)
        if json_match:
            return json.loads(json_match.group())
        else:
            return None
    except Exception as e:
        print("JSON parsing error:", e)
        return None


# -----------------------------
# 3. VALIDATION
# -----------------------------
def validate(data):
    errors = []

    if not data:
        errors.append("No data parsed")

    else:
        if not data.get("stock"):
            errors.append("Missing stock")

        if len(data.get("inflow", [])) == 0:
            errors.append("No inflow")

        if len(data.get("outflow", [])) == 0:
            errors.append("No outflow")

    return errors


# -----------------------------
# 4. GENERATE MDL
# -----------------------------
def generate_mdl(data):
    stock = data["stock"]
    inflow = " + ".join(data["inflow"])
    outflow = " + ".join(data["outflow"])

    equation = f"{stock} = INTEG({inflow} - {outflow}, 1000)"

    return equation


# -----------------------------
# 5. FULL PIPELINE
# -----------------------------
def run_pipeline(user_input):
    print("🔹 Input:", user_input)

    raw = call_llm(user_input)
    print("\n🔹 Raw LLM Output:\n", raw)

    parsed = parse_json(raw)
    print("\n🔹 Parsed JSON:\n", parsed)

    errors = validate(parsed)

    if errors:
        print("\n❌ Validation Errors:", errors)
        return None

    mdl = generate_mdl(parsed)
    print("\n✅ MDL Output:\n", mdl)

    return parsed, mdl